# Nyung: 한국인만 읽을 수 있는 욕 감지기
**지금까지 한 것: 베이스라인 평가 (E1) + 데이터 준비**

| 섹션 | 내용 |
|---|---|
| 0 | 설치, 드라이브, 경로 |
| 1 | 모듈 파일 (`normalize.py`, `eval_baseline.py`) |
| 2 | UnSmile EDA |
| 3 | 베이스라인 평가 (UnSmile valid, 프로브, KOTOX, 속도) |
| 4 | AI Hub 불러오기 → 라벨 변환 → 정제 → 분할 → 베이스라인 평가 |
| 5 | 말장난형 우회 욕 세트 |

> 런타임을 새로 시작하면 **0 → 1 순서로 먼저 실행**한 뒤 필요한 섹션을 실행합니다.

## 0. 설정

In [ ]:
# 패키지 설치 (코랩 기본 패키지 외에 필요한 것만)
!pip install -q "transformers>=4.38.0" "accelerate>=0.28.0" "scikit-learn>=1.3.0" koreanize-matplotlib

In [ ]:
# 드라이브 연결 + 공통 import + 경로
from google.colab import drive
drive.mount("/content/drive")

import os, sys, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib                      # 그래프 한글 깨짐 방지
from sklearn.model_selection import train_test_split

sys.path.append("/content")                      # normalize.py, eval_baseline.py 위치
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 100)

# AI Hub 원본 (드라이브, 깃허브에 올리지 않음)
BASE = "/content/drive/MyDrive/Nyung/유해표현 검출 AI모델 학습용 데이터/1.데이터"
# 전처리 결과 저장 위치
OUT = "/content/drive/MyDrive/Nyung/data/processed"
os.makedirs(OUT, exist_ok=True)

## 1. 모듈 파일
`%%writefile`은 셀의 **첫 줄**에 있어야 합니다. 이 셀들은 실행해도 코드가 돌지 않고 파일로 저장만 됩니다.

In [ ]:
%%writefile /content/normalize.py
"""
정규화: 비틀어 쓴 표기를 되돌림 (의미 판단은 하지 않음, 의미는 모델 몫)
level: light / strict / strict_hy (strict + 한영 변환)
"""
import re

CHO  = "ㄱㄲㄴㄷㄸㄹㅁㅂㅃㅅㅆㅇㅈㅉㅊㅋㅌㅍㅎ"
JUNG = "ㅏㅐㅑㅒㅓㅔㅕㅖㅗㅘㅙㅚㅛㅜㅝㅞㅟㅠㅡㅢㅣ"
JONG = "ㄱㄲㄳㄴㄵㄶㄷㄹㄺㄻㄼㄽㄾㄿㅀㅁㅂㅄㅅㅆㅇㅈㅊㅋㅌㅍㅎ"
HANGUL = "가-힣ㄱ-ㅣ"

# 보이지 않는 문자 (zero-width space 등)
ZERO_WIDTH = re.compile(r"[​-‏⁠﻿­]")
# light: 한글 사이에 끼운 숫자·기호 제거 (시1발 → 시발), 띄어쓰기는 건드리지 않음
NOISE_LIGHT  = re.compile(rf"(?<=[{HANGUL}])[0-9!@#$%^&*.,~\-_+=|/\\:;'\"`?<>()\[\]{{}}]+(?=[{HANGUL}])")
# strict: 영문 1~2글자 끼우기도 제거 (시a발 → 시발)
NOISE_STRICT = re.compile(rf"(?<=[{HANGUL}])[0-9A-Za-z!@#$%^&*.,~\-_+=|/\\:;'\"`?<>()\[\]{{}}]{{1,2}}(?=[{HANGUL}])")
# 자음 뒤에 모음 ㅣ 대신 쓴 1, l, I, |, ! (ㅅl발 → ㅅㅣ발)
I_LOOKALIKE = re.compile(r"(?<=[ㄱ-ㅎ])[1lI|!](?=[가-힣ㄱ-ㅣ]|$)")

# 한글 자판으로 친 영문 → 한글 자모 (2벌식)
KEY = dict(zip("qwertyuiopasdfghjklzxcvbnm", "ㅂㅈㄷㄱㅅㅛㅕㅑㅐㅔㅁㄴㅇㄹㅎㅗㅓㅏㅣㅋㅌㅊㅍㅠㅜㅡ"))
KEY.update(dict(zip("QWERTOP", "ㅃㅉㄸㄲㅆㅒㅖ")))


def _fullwidth_ascii(s):
    """전각 영숫자·기호만 반각으로 (한글은 그대로, NFKC는 한글 자모를 바꿔서 사용 안 함)"""
    return "".join(chr(ord(c) - 0xFEE0) if 0xFF01 <= ord(c) <= 0xFF5E else c for c in s)


def compose_jamo(s):
    """흩어진 자모를 음절로: ㅅㅣ발 → 시발"""
    out, i = [], 0
    while i < len(s):
        if i + 1 < len(s) and s[i] in CHO and s[i + 1] in JUNG:
            c, v, t = CHO.index(s[i]), JUNG.index(s[i + 1]), 0
            i += 2
            # 다음 글자가 받침 후보이고, 그 뒤가 모음이 아니면 받침으로 붙임
            if i < len(s) and s[i] in JONG and not (i + 1 < len(s) and s[i + 1] in JUNG):
                t = JONG.index(s[i]) + 1
                i += 1
            out.append(chr(0xAC00 + (c * 21 + v) * 28 + t))
        else:
            out.append(s[i]); i += 1
    return "".join(out)


def fix_hanyoung(text):
    """한영 전환 안 하고 친 글자 복원: tlqkf → 시발, qudtls → 병신
    영문 3글자 이상이 '완전한 한글 음절'로만 바뀔 때만 변환 (game, hello 등은 대부분 그대로)"""
    def conv(m):
        out = compose_jamo("".join(KEY.get(c, KEY.get(c.lower(), c)) for c in m.group()))
        return out if re.fullmatch(r"[가-힣]+", out) else m.group()
    return re.sub(r"[A-Za-z]{3,}", conv, text)


def _vowel(ch):
    code = ord(ch) - 0xAC00
    return None if not 0 <= code < 11172 else (code // 28) % 21


def _has_jong(ch):
    code = ord(ch) - 0xAC00
    return 0 <= code < 11172 and code % 28 != 0


def drop_elongation(s):
    """시이이발 → 시발: 받침 없는 음절 뒤에 같은 모음의 'ㅇ+모음'이 붙은 늘여 쓰기 제거"""
    out = []
    for ch in s:
        code = ord(ch) - 0xAC00
        is_o_vowel = 0 <= code < 11172 and code // 588 == CHO.index("ㅇ") and code % 28 == 0
        if out and is_o_vowel and not _has_jong(out[-1]) and _vowel(out[-1]) is not None \
                and _vowel(out[-1]) == _vowel(ch):
            continue
        out.append(ch)
    return "".join(out)


def collapse_repeats(s, keep=2):
    """같은 글자 반복을 keep개로: ㅋㅋㅋㅋ → ㅋㅋ, !!!!! → !!"""
    return re.sub(r"(.)\1{%d,}" % keep, lambda m: m.group(1) * keep, s)


def join_spaced_syllables(s):
    """한 글자씩 띄어 쓴 구간 붙이기: 시 발 놈 아 → 시발놈아
    (알려진 한계: '놈 또', '야 이' 같은 정상 한 글자 단어도 붙음)"""
    return re.sub(rf"(?<![{HANGUL}])([{HANGUL}](?: [{HANGUL}])+)(?![{HANGUL}])",
                  lambda m: m.group(1).replace(" ", ""), s)


def normalize(text, level="strict"):
    strict = level in ("strict", "strict_hy")
    s = ZERO_WIDTH.sub("", str(text))
    s = _fullwidth_ascii(s)
    s = re.sub(r"\s+", " ", s).strip()
    if level == "strict_hy":                 # 한영 변환은 영문 끼우기 제거보다 먼저
        s = fix_hanyoung(s)
    if strict:
        s = I_LOOKALIKE.sub("ㅣ", s)
        s = NOISE_STRICT.sub("", s)
    else:
        s = NOISE_LIGHT.sub("", s)
    s = compose_jamo(s)
    s = collapse_repeats(s, keep=2)
    if strict:
        s = drop_elongation(s)
        s = join_spaced_syllables(s)
    return s

In [ ]:
%%writefile /content/eval_baseline.py
"""
베이스라인 평가: UnSmile 모델을 바꾸지 않고 그대로 썼을 때 욕설을 얼마나 잡는지 측정
  evaluate(df, norm, score_type)  text,label DataFrame 평가
  run_kotox() / run_csv(path)     KOTOX, CSV 평가 세트 일괄 평가
  show_errors(df)                 놓친 욕(FN) / 잘못 막은 정상(FP) 예시
  demo([...]) / latency()         문장 바로 넣어 보기 / 속도 측정
"""
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix,
                             roc_auc_score, average_precision_score)
from normalize import normalize

NAME = "smilegate-ai/kor_unsmile"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 64                                   # AI Hub 문장의 94%가 64토큰 이내, 채팅은 더 짧음
SHOW = ["set", "norm", "score", "n", "precision", "recall", "f1", "fpr", "pr_auc"]

tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForSequenceClassification.from_pretrained(NAME).to(DEVICE).eval()

# 라벨 위치 (config에 이름이 없으면 README 순서: 8 = 악플/욕설, 9 = clean)
id2label = {int(k): v for k, v in model.config.id2label.items()}
CURSE_IDX = next((i for i, l in id2label.items() if "욕설" in l), 8)
CLEAN_IDX = next((i for i, l in id2label.items() if l.lower() == "clean"), 9)
print(f"id2label: {id2label}\n악플/욕설 = {CURSE_IDX}, clean = {CLEAN_IDX}, device = {DEVICE}")


def _norm(texts, norm):
    """norm: none / light / strict / strict_hy"""
    texts = [str(t) for t in texts]
    return texts if norm == "none" else [normalize(t, level=norm) for t in texts]


@torch.inference_mode()
def predict_probs(texts, bs=64):
    """문장 리스트 → (N, 10) 라벨별 확률 (다중 라벨이라 sigmoid)"""
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], padding=True, truncation=True,
                  max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            logits = model(**enc).logits
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)


def scores_from(probs, score_type="curse"):
    """curse: 악플/욕설 확률만 봄 / any_hate: 1 - clean (어떤 혐오든 걸리면 차단)"""
    return probs[:, CURSE_IDX] if score_type == "curse" else 1 - probs[:, CLEAN_IDX]


def score_texts(texts, norm="none", score_type="curse"):
    return scores_from(predict_probs(_norm(texts, norm)), score_type)


def evaluate(df, norm="none", score_type="curse", threshold=0.5, name=""):
    """df: text, label(0/1) 열"""
    y = df["label"].astype(int).values
    s = score_texts(df["text"], norm, score_type)
    p = (s >= threshold).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    both = len(set(y)) == 2                    # 정답이 한 종류뿐이면 AUC 계산 불가
    return {
        "set": name, "norm": norm, "score": score_type, "n": len(y),
        "precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4),
        "fpr": round(fp / (fp + tn), 4) if (fp + tn) else None,   # 정상을 욕으로 막은 비율
        "roc_auc": round(roc_auc_score(y, s), 4) if both else None,
        "pr_auc": round(average_precision_score(y, s), 4) if both else None,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }


def run_eval(df, name, norms=("none", "strict"), score_types=("curse",)):
    """같은 세트를 정규화 방식 × 판정 방식 조합으로 평가해 표로 출력"""
    res = pd.DataFrame([evaluate(df, nm, st, name=name) for nm in norms for st in score_types])
    print(res[SHOW].to_string(index=False))
    return res


KOTOX = "https://huggingface.co/datasets/ssgyejin/KOTOX-classification/resolve/main/data"

def load_kotox(level="total", split="test"):
    return pd.read_csv(f"{KOTOX}/{level}/{split}.csv")


def run_kotox(levels=("easy", "normal", "hard", "total"), norms=("none", "strict"), score_types=("curse",)):
    res = pd.concat([run_eval(load_kotox(lv, "test"), f"KOTOX-{lv}", norms, score_types) for lv in levels])
    return res.reset_index(drop=True)


def run_csv(path, name, norms=("none", "strict"), score_types=("curse",)):
    return run_eval(pd.read_csv(path), name, norms, score_types)


def show_errors(df, norm="strict", score_type="curse", threshold=0.5, n=10):
    """놓친 욕(FN)과 잘못 막은 정상(FP) 예시"""
    inp = _norm(df["text"], norm)
    d = df.assign(normalized=inp, score=scores_from(predict_probs(inp), score_type))
    fn = d[(d.label == 1) & (d.score < threshold)].sort_values("score")
    fp = d[(d.label == 0) & (d.score >= threshold)].sort_values("score", ascending=False)
    print(f"--- 놓친 욕 (FN) {len(fn)}개 중 {min(n, len(fn))}개 ---")
    print(fn[["text", "normalized", "score"]].head(n).to_string(index=False))
    print(f"\n--- 잘못 막은 정상 (FP) {len(fp)}개 중 {min(n, len(fp))}개 ---")
    print(fp[["text", "normalized", "score"]].head(n).to_string(index=False))
    return fn, fp


def demo(texts, norm="strict"):
    """문장 몇 개를 바로 넣어 보기"""
    inp = _norm(texts, norm)
    for t, n_, pr in zip(texts, inp, predict_probs(inp)):
        print(f"{t!r:28} → {n_!r:24} 욕설 {pr[CURSE_IDX]:.2f} | 혐오(1-clean) {1 - pr[CLEAN_IDX]:.2f}")


@torch.inference_mode()
def latency(sample=("야 이 시1발 놈아", "오늘 날씨 좋네", "ㅅㅣ발 진짜"), runs=200, norm="strict"):
    """한 문장 처리 시간 (정규화 + 토큰화 + 모델), ms"""
    for _ in range(10):                        # 예열 (첫 실행은 느림)
        predict_probs(_norm(sample[:1], norm))
    times = []
    for i in range(runs):
        if DEVICE == "cuda": torch.cuda.synchronize()   # GPU 작업이 끝난 시점까지 재기
        t0 = time.perf_counter()
        predict_probs(_norm([sample[i % len(sample)]], norm))
        if DEVICE == "cuda": torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    gpu = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
    p50, p95, p99 = np.percentile(times, [50, 95, 99])
    print(f"[{gpu}] p50 {p50:.1f}ms | p95 {p95:.1f}ms | p99 {p99:.1f}ms")
    return times

In [ ]:
# 모듈 불러오기 (eval_baseline을 import하면 UnSmile 모델을 GPU에 올림, 처음엔 10~30초)
from normalize import normalize
from eval_baseline import *

# 정규화 동작 확인
for t, lv in [("tlqkf 진짜", "strict_hy"), ("ㅅl발", "strict"), ("씨이이이발", "strict"),
              ("시 발 놈 아", "strict"), ("수박 씨 발라 먹자", "strict"), ("game 하자", "strict_hy")]:
    print(f"{lv:9} {t!r:14} → {normalize(t, lv)!r}")

## 2. UnSmile EDA
베이스 모델이 어떤 데이터로 학습됐는지 먼저 확인합니다.

In [ ]:
# UnSmile 원본 데이터 (깃허브 raw TSV)
RAW = "https://raw.githubusercontent.com/smilegate-ai/korean_unsmile_dataset/main"
us_train = pd.read_csv(f"{RAW}/unsmile_train_v1.0.tsv", sep="\t")
us_valid = pd.read_csv(f"{RAW}/unsmile_valid_v1.0.tsv", sep="\t")

for d in (us_train, us_valid):                 # 열 이름 공백 정리 ("기타 혐오" → "기타혐오")
    d.columns = [c.replace(" ", "") for c in d.columns]

HATE = ["여성/가족", "남성", "성소수자", "인종/국적", "연령", "지역", "종교", "기타혐오"]
LABELS = HATE + ["악플/욕설", "clean"]

def add_group(d):
    """다중 라벨을 그룹 하나로 요약: 욕설만 / 욕설+혐오 / 혐오만 / clean"""
    curse, hate = d["악플/욕설"] == 1, d[HATE].sum(axis=1) > 0
    d["group"] = np.select([curse & ~hate, curse & hate, ~curse & hate, d["clean"] == 1],
                           ["욕설만", "욕설+혐오", "혐오만", "clean"], default="기타")
    d["n_labels"] = d[LABELS].sum(axis=1)
    return d

us_train, us_valid = add_group(us_train), add_group(us_valid)
print(us_train.shape, us_valid.shape)

In [ ]:
# EDA 함수 (AI Hub에도 재사용)
def eda_structure(df, name):
    print(f"===== [{name}] 구조 =====")
    print(f"행 {df.shape[0]:,}개 / 열 {df.shape[1]}개"); print(df.dtypes.to_string())
    display(df.head())

def eda_missing(df, name, text_col="text"):
    print(f"\n===== [{name}] 결측치 =====")
    na = df.isna().sum()
    print(pd.DataFrame({"결측 개수": na, "비율(%)": (na / len(df) * 100).round(2)}).to_string())
    blank = df[text_col].astype(str).str.strip().eq("").sum()
    print(f"빈 문자열 문장: {blank:,}개")

def eda_duplicates(df, name, text_col="text", label_col="label"):
    print(f"\n===== [{name}] 중복 =====")
    conflict = df.groupby(text_col)[label_col].nunique().loc[lambda s: s > 1]
    print(f"완전히 같은 행: {df.duplicated().sum():,}개")
    print(f"문장만 같은 행: {df.duplicated(subset=[text_col]).sum():,}개")
    print(f"같은 문장인데 라벨이 다름: {len(conflict):,}개")

def check_overlap(a, b, name_a, name_b, text_col="text"):
    """train / test 사이 같은 문장 = 데이터 누수"""
    n = len(set(a[text_col]) & set(b[text_col]))
    print(f"[누수 확인] {name_a} ∩ {name_b} 같은 문장: {n:,}개")
    return n

def eda_labels(df, name, label_col="label"):
    """이진 라벨 개수와 비율"""
    vc = df[label_col].value_counts().sort_index()
    print(f"\n===== [{name}] 라벨 분포 =====")
    print(pd.DataFrame({"개수": vc, "비율(%)": (vc / vc.sum() * 100).round(2)}).to_string())
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    vc.plot.bar(ax=ax[0], color=["#4C78A8", "#E45756"][:len(vc)]); ax[0].set_title(f"{name} 라벨 개수")
    for i, v in enumerate(vc): ax[0].text(i, v, f"{v:,}", ha="center", va="bottom")
    ax[1].pie(vc, labels=vc.index, autopct="%.1f%%", colors=["#4C78A8", "#E45756"][:len(vc)])
    ax[1].set_title(f"{name} 라벨 비율"); plt.tight_layout(); plt.show()

def eda_multilabel(d, name):
    """UnSmile 다중 라벨: 라벨별 개수 / 그룹 비율 / 문장당 라벨 수"""
    counts = d[LABELS].sum().sort_values(ascending=False)
    print(f"\n===== [{name}] 라벨 분포 =====")
    print(pd.DataFrame({"개수": counts, "비율(%)": (counts / len(d) * 100).round(2)}).to_string())
    g = d["group"].value_counts()
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
    counts.plot.bar(ax=ax[0], color="#4C78A8"); ax[0].set_title(f"{name} 라벨별 문장 수")
    ax[1].pie(g, labels=g.index, autopct="%.1f%%"); ax[1].set_title(f"{name} 그룹 비율")
    d["n_labels"].value_counts().sort_index().plot.bar(ax=ax[2], color="#E45756")
    ax[2].set_title(f"{name} 문장당 라벨 개수"); plt.tight_layout(); plt.show()

def eda_length(df, name, text_col="text", label_col="label", tokenizer=None):
    """글자 수 / 토큰 수 분포 (max_len 결정 근거)"""
    d = df.copy()
    d["char_len"] = d[text_col].astype(str).str.len()
    cols = ["char_len"]
    if tokenizer is not None:
        d["tok_len"] = [len(tokenizer.tokenize(t)) for t in d[text_col].astype(str)]
        cols.append("tok_len")
    print(f"\n===== [{name}] 길이 분포 =====")
    for c in cols:
        p = np.percentile(d[c], [50, 90, 95, 99])
        print(f"{c}: 평균 {d[c].mean():.1f} | 중앙 {p[0]:.0f} | 90% {p[1]:.0f} | 95% {p[2]:.0f} | 99% {p[3]:.0f} | 최대 {d[c].max()}")
    fig, axes = plt.subplots(1, len(cols), figsize=(6 * len(cols), 4))
    for ax, c in zip(np.atleast_1d(axes), cols):
        for lab, g in d.groupby(label_col):
            ax.hist(g[c], bins=50, alpha=0.6, label=f"label {lab}")
        ax.axvline(np.percentile(d[c], 95), color="k", ls="--", label="95%")
        ax.set_title(f"{name} {c} 분포"); ax.legend()
    plt.tight_layout(); plt.show()
    return d

In [ ]:
# 구조, 결측, 중복, 누수
eda_structure(us_train, "UnSmile train")
eda_missing(us_train, "UnSmile train", text_col="문장")
eda_duplicates(us_train, "UnSmile train", text_col="문장", label_col="group")
check_overlap(us_train, us_valid, "train", "valid", text_col="문장")

In [ ]:
# 라벨 분포 (다중 라벨)
eda_multilabel(us_train, "UnSmile train")
eda_multilabel(us_valid, "UnSmile valid")

In [ ]:
# 길이 분포 (욕설 여부로 나눠서)
us_train["is_curse"] = us_train["악플/욕설"]
_ = eda_length(us_train, "UnSmile train", text_col="문장", label_col="is_curse", tokenizer=tok)

## 3. 베이스라인 평가 (E1)
UnSmile을 **그대로** 썼을 때의 점수입니다. 학습 후(E2, E3)에 같은 세트로 다시 재서 비교합니다.

In [ ]:
# UnSmile valid: 정규화 방식별 비교 (정답 = 악플/욕설)
valid_eval = us_valid.rename(columns={"문장": "text"}).assign(label=us_valid["악플/욕설"])
res_unsmile = run_eval(valid_eval, "UnSmile valid", norms=("none", "light", "strict", "strict_hy"))

In [ ]:
# 문장 바로 넣어 보기: 우회 욕 / 정상 / 오탐 함정
demo(["시1발 진짜", "ㅅㅣ발 놈아", "씨이이발", "시 발 놈 아", "tlqkf",
      "오늘 날씨 좋네", "시발점부터 보자", "수박 씨 발라 먹자"], norm="strict_hy")

In [ ]:
# 프로브: 유형별로 직접 만든 확인용 문장 (강조 비속어는 욕이 아니라고 판단해 제외)
probe = pd.DataFrame([
    # A. 정규화로 풀리는 우회 (욕)
    ("시1발 진짜", 1, "A_정규화"), ("ㅅㅣ발 놈아", 1, "A_정규화"), ("시 발 놈 아", 1, "A_정규화"),
    ("병1신같네", 1, "A_정규화"), ("씨이이발", 1, "A_정규화"),
    # B. 초성만 (욕)
    ("ㅅㅂ 짜증나", 1, "B_초성"), ("ㅆㅂ 뭐하냐", 1, "B_초성"), ("ㅂㅅ아", 1, "B_초성"),
    ("ㅁㅊㄴ", 1, "B_초성"), ("ㅈㄴ 못하네", 1, "B_초성"),
    # C. 된소리·모음 비틀기 (욕)
    ("씨빨 진짜", 1, "C_비틀기"), ("쉬바 뭐냐", 1, "C_비틀기"), ("시바 꺼져", 1, "C_비틀기"),
    ("십알놈", 1, "C_비틀기"), ("븅신아", 1, "C_비틀기"), ("빙신같은게", 1, "C_비틀기"),
    ("미틴놈", 1, "C_비틀기"), ("닭쳐라", 1, "C_비틀기"), ("개새기", 1, "C_비틀기"), ("개쉐이", 1, "C_비틀기"),
    # D. 한영 변환 (욕)
    ("tlqkf", 1, "D_한영"), ("qudtls", 1, "D_한영"), ("c8 진짜", 1, "D_한영"), ("sibal", 1, "D_한영"),
    # E. 단어 바꿔치기·은어 (욕)
    ("느금마", 1, "E_은어"), ("니애미", 1, "E_은어"), ("몽뭉이 같은 놈", 1, "E_은어"), ("엠창", 1, "E_은어"),
    # F. 욕 없는 모욕
    ("머리는 장식이냐", 1, "F_모욕"), ("너 같은 건 왜 사냐", 1, "F_모욕"), ("부모님이 불쌍하다", 1, "F_모욕"),
    # G. 오탐 함정 (정상)
    ("시발점부터 보자", 0, "G_함정"), ("다음 역이 시발역이야", 0, "G_함정"), ("수박 씨 발라 먹자", 0, "G_함정"),
    ("내 십팔번은 이 노래", 0, "G_함정"), ("병신년은 2016년이다", 0, "G_함정"), ("새끼 고양이 귀여워", 0, "G_함정"),
    ("다음 주 개강이야", 0, "G_함정"), ("18세 이상 관람가", 0, "G_함정"), ("채팅에서 욕하지 마세요", 0, "G_함정"),
    ("닥쳐올 미래가 걱정이다", 0, "G_함정"), ("ㅎㅏㅇ상 고마워", 0, "G_함정"), ("우리 집 개 산책 시켜야 해", 0, "G_함정"),
], columns=["text", "label", "type"])

In [ ]:
# 프로브 평가: 유형별 정답률 (정규화 방식별)
NORMS = ("none", "strict", "strict_hy")
for nm in NORMS:
    s = score_texts(probe["text"], nm, "curse")
    probe[f"score_{nm}"] = s.round(2)
    probe[f"ok_{nm}"] = (s >= 0.5).astype(int) == probe["label"]

print(probe.groupby("type")[[f"ok_{nm}" for nm in NORMS]].mean().round(2))

# 틀린 문장만 (가장 강한 정규화 기준)
display(probe.loc[~probe["ok_strict_hy"], ["type", "text", "label"] + [f"score_{nm}" for nm in NORMS]])

In [ ]:
# KOTOX test (난이도별, 비틀어 쓴 욕 평가)
res_kotox = run_kotox(norms=NORMS)

In [ ]:
# 속도: 한 문장 처리 시간 (정규화 + 모델)
_ = latency(norm="strict_hy")

## 4. AI Hub 유해표현 데이터
- stage1: 유해 여부 (content, label)
- stage2: 유형 11개 + 정상 (값 0~3 = 강도)

**라벨 방침:** 욕설·모욕 > 0 → 1 / 정상만 → 0 / 다른 혐오만 → 따로 보관 (학습에 0으로 넣을지는 학습 단계에서 결정)

In [ ]:
# 불러오기
s1 = pd.read_csv(f"{BASE}/1. 유해표현 검출/stage1_data.csv")
s2 = pd.read_csv(f"{BASE}/2. 유해 카테고리 분류/stage2_data.csv")
print("stage1", s1.shape, "| stage2", s2.shape)
display(s1.head(3)); display(s2.head(3))

In [ ]:
# 분포와 문제 확인
CATS = list(s2.columns[1:])                     # 문장 다음 열부터 유형
for c in CATS:
    print(f"{c:12}", s2[c].value_counts().sort_index().to_dict())

print("\nstage1 라벨:", s1.label.value_counts().to_dict())
print("두 파일 겹치는 문장:", len(set(s1.content) & set(s2["문장"])), "→ stage2만 사용")
print("'' 유해 구간 표시 비율  s1:", s1.content.str.contains("''").mean().round(3),
      "| s2:", s2["문장"].str.contains("''").mean().round(3))
print("#@ 비식별 태그 비율 s1:", s1.content.str.contains("#@").mean().round(3))

In [ ]:
# 라벨 변환
OTHER = [c for c in CATS if c not in ("모욕", "욕설", "정상")]   # 욕설·모욕 외 혐오 유형

def clean(t):
    t = re.sub(r"'{2,}", "", str(t))            # 유해 구간 표시 '' ''' 제거 (지름길 학습 방지)
    t = re.sub(r"#@[^#]*#", "OO", t)            # #@이름# 같은 비식별 태그 → OO
    t = t.replace("\\", "").strip('"')
    return re.sub(r"\s+", " ", t).strip()

ah = s2.rename(columns={"문장": "raw"})
pos   = (ah["모욕"] > 0) | (ah["욕설"] > 0)                          # 욕설·모욕 → 1
neg   = (ah["정상"] == 1) & (ah[OTHER].sum(axis=1) == 0) & ~pos     # 정상만 → 0
other = ~pos & (ah["정상"] == 0)                                     # 다른 혐오만
ah["label"] = np.select([pos, neg], [1, 0], default=-1)              # -1 = 학습 라벨에서 제외
print("변환:", ah["label"].value_counts().to_dict(), "| 다른 혐오만:", other.sum())

# 유해 구간 표시가 욕 쪽에만 몰려 있는지 (몰려 있으면 지우는 게 맞다는 근거)
print("'' 표시 비율 (라벨별):", ah.groupby("label")["raw"].apply(lambda s: s.str.contains("''").mean().round(3)).to_dict())
print("\n욕설 강도 1 샘플:"); print(ah.loc[ah["욕설"] == 1, "raw"].sample(8, random_state=0).to_string())

In [ ]:
# 정제: 표시·태그 제거 → 짧은 문장 제거 → 라벨 충돌 제거 → 중복 제거 → 평가 세트와 겹치는 문장 제거
def tidy(d):
    d = d.assign(text=d["raw"].map(clean))
    d = d[d["text"].str.len() >= 2]
    conflict = d.groupby("text")["label"].nunique().loc[lambda s: s > 1].index
    return d[~d["text"].isin(conflict)].drop_duplicates("text")[["text", "label"]]

# 평가에 쓰는 문장 (학습에 들어가면 점수가 부풀려짐)
EVAL_TEXTS = set(valid_eval["text"]) | set(load_kotox("total", "test")["text"]) | set(probe["text"])

df  = tidy(ah[ah["label"] >= 0])
oth = tidy(ah[other].assign(label=0))
df  = df[~df["text"].isin(EVAL_TEXTS)]
oth = oth[~oth["text"].isin(EVAL_TEXTS | set(df["text"]))]
print("정제 후:", len(df), df["label"].value_counts().to_dict(), "| 다른 혐오:", len(oth))
eda_labels(df, "AI Hub 정제 후")

In [ ]:
# 토큰 길이: MAX_LEN 64로 충분한지 (5,000개 샘플)
tl = df["text"].sample(5000, random_state=0).map(lambda t: len(tok(t)["input_ids"]))
print(tl.describe(percentiles=[.5, .9, .95, .99]).round(1).to_dict())
print("64 초과 비율:", (tl > 64).mean().round(3))

In [ ]:
# 분할 8:1:1 (중복을 먼저 지웠으므로 같은 문장이 train과 test에 함께 들어가지 않음) → 드라이브 저장
def split_save(d, prefix, stratify=True):
    st = d["label"] if stratify else None
    tr, tmp = train_test_split(d, test_size=0.2, stratify=st, random_state=42)
    va, te = train_test_split(tmp, test_size=0.5, stratify=tmp["label"] if stratify else None, random_state=42)
    for name, part in [("train", tr), ("valid", va), ("test", te)]:
        part.to_csv(f"{OUT}/{prefix}_{name}.csv", index=False)
    print(f"{prefix}: train {len(tr)} / valid {len(va)} / test {len(te)} | 1 비율 {tr['label'].mean():.3f}")
    return tr, va, te

tr, va, te = split_save(df, "aihub")                          # 욕설·모욕 vs 정상
o_tr, o_va, o_te = split_save(oth, "aihub_other", stratify=False)   # 다른 혐오 (전부 0)

In [ ]:
# 베이스라인을 AI Hub test에서 측정
res_aihub = run_eval(te, "AIHub-test", norms=("none", "strict"), score_types=("curse", "any_hate"))

# 다른 혐오 (전부 정답 0): FPR만 의미 있음. 높을수록 욕 아닌 문장을 많이 막는다는 뜻
res_other = run_eval(o_te, "AIHub-다른혐오", norms=("none",), score_types=("curse", "any_hate"))

## 5. 말장난형 우회 욕
수박씨 발라먹을 놈, 조카 18색 크레파스 같은 표현. 정규화로는 풀 수 없고(띄어쓰기를 지우면 정상 문장도 걸림), 학습으로 문맥을 가르쳐야 합니다.

In [ ]:
# 기존 데이터에 말장난 표현이 얼마나 있는지 (AI Hub + KOTOX)
PUN_WORDS = ["수박씨", "조카크레파스", "조까", "십장생", "시베리아", "신발끈", "개나리", "18색", "씨발라"]

kotox = pd.concat([load_kotox("total", sp) for sp in ("train", "valid", "test")])
ah_flat = df["text"].str.replace(" ", "")
kx_norm = kotox["text"].astype(str).map(lambda t: normalize(t, "strict_hy").replace(" ", ""))

rows = []
for w in PUN_WORDS:
    a, k = df[ah_flat.str.contains(w)], kotox[kx_norm.str.contains(w)]
    rows.append({"표현": w, "AI Hub": len(a), "AI Hub 욕 비율": round(a.label.mean(), 2) if len(a) else None,
                 "KOTOX": len(k)})
display(pd.DataFrame(rows))

In [ ]:
# 말장난 대조쌍 생성: (욕으로 쓰는 형태들, 원래 뜻의 정상 문장들)
PUNS = {
    "수박씨":      (["수박씨 발라먹을", "수박씨 발라버릴"],
                    ["수박씨 발라 먹자", "수박씨는 발라내고 줘", "수박씨 뱉지 말고 버려"]),
    "시베리아":    (["시베리아", "시베리아 벌판에서 귤이나 깔"],
                    ["시베리아는 정말 춥대", "시베리아 횡단열차 타보고 싶다"]),
    "십장생":      (["십장생", "십장생 같은"],
                    ["십장생은 장수를 상징하는 열 가지야", "민화에 십장생이 그려져 있어"]),
    "신발끈":      (["신발끈", "신발끈 같은"],
                    ["신발끈이 풀렸어", "신발끈 좀 묶고 갈게"]),
    "조카크레파스": (["조카크레파스", "조카 18색 크레파스", "조카신발"],
                    ["조카 크레파스 사줬어", "조카가 18색 크레파스로 그림 그렸어"]),
    "개나리":      (["개나리", "개나리 같은"],
                    ["개나리가 활짝 폈다", "봄이라 개나리 보러 가자"]),
    "시바":        (["시바", "시바 같은"],
                    ["시바견 너무 귀엽다", "우리 집 시바 산책 시켜야 해"]),
}

# 문장 틀: 학습용과 평가용을 겹치지 않게 나눔
TRAIN_T     = ["{} 놈", "이런 {} 같은 새끼", "{} 진짜 왜 저래", "너 진짜 {}이다", "아 {} 짜증나 죽겠네", "{} 새끼야 꺼져"]
EVAL_T      = ["저 {} 같은 놈 또 왔네", "{} 놈아 입 닥쳐", "야 이 {}아"]          # 모욕어 포함
EVAL_T_HARD = ["아 {}", "{}!! 진짜", "{}이네 ㅋㅋ", "{} 하고 앉아있네"]          # 표현만 (말장난을 진짜 읽는지)

# 정상 문장: 표현마다 마지막 하나는 평가용으로 남김
NEG_TR = [s for _, normals in PUNS.values() for s in normals[:-1]]
NEG_EV = [normals[-1] for _, normals in PUNS.values()]

def build(templates, normals):
    """욕 형태 × 문장 틀 = 1, 원래 뜻 문장 = 0"""
    pos = [t.format(f) for forms, _ in PUNS.values() for f in forms for t in templates]
    return pd.DataFrame({"text": pos + normals, "label": [1] * len(pos) + [0] * len(normals)})

pun_sets = {"pun_train": build(TRAIN_T, NEG_TR),
            "pun_eval": build(EVAL_T, NEG_EV),
            "pun_eval_hard": build(EVAL_T_HARD, NEG_EV)}
for name, d in pun_sets.items():
    d.to_csv(f"{OUT}/{name}.csv", index=False)
    print(f"{name:14}", d["label"].value_counts().to_dict())

In [ ]:
# 베이스라인을 말장난 평가 세트에서 측정
res_pun = pd.concat([run_eval(pun_sets["pun_eval"], "Pun-eval", norms=("strict",)),
                     run_eval(pun_sets["pun_eval_hard"], "Pun-hard", norms=("strict",))])

In [ ]:
# 모델이 말장난을 읽는지, "놈" 같은 단어만 보는지 확인
demo(["저 수박씨 발라먹을 같은 놈 또 왔네", "저 철수 같은 놈 또 왔네",      # 표현 있음 vs 없음
      "수박씨 발라먹을", "조카 18색 크레파스", "십장생", "시베리아 벌판에서 귤이나 깔"])   # 모욕어 없이 표현만
_ = show_errors(pun_sets["pun_eval"], norm="strict")